# Raman Alignment Prediction Pipeline

In [1]:
from __future__ import annotations

import os
from pathlib import Path

if os.environ.get("MPLCONFIGDIR") is None:
    _mpl_dir = Path.cwd() / ".mplconfig"
    _mpl_dir.mkdir(parents=True, exist_ok=True)
    os.environ["MPLCONFIGDIR"] = str(_mpl_dir)

import ast
import json
import math
import re
import sys
import time
import warnings
from pathlib import Path
from typing import Any, Dict, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
from IPython.display import Image, Markdown, display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings(
    "ignore",
    message="The TorchScript type system doesn't support instance-level annotations",
)


In [2]:
# Resolve repo root and import project modules.
if (Path.cwd() / "ramanchembl_pipeline").exists():
    REPO_ROOT = Path.cwd()
else:
    REPO_ROOT = Path.cwd().parent

CAPSULE_CODE = REPO_ROOT / "capsule-3259363" / "code"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(CAPSULE_CODE) not in sys.path:
    sys.path.insert(0, str(CAPSULE_CODE))

from train.train_detanet import build_model  # noqa: E402
from detanet_model.constant import atom_masses  # noqa: E402
from detanet_model.model_loader import Hi_model, Hij_model  # noqa: E402
from detanet_model.spectra_simulator import (  # noqa: E402
    Lorenz_broadening,
    chain_rule_raman,
    get_raman_act,
    get_raman_intensity,
    hessfreq,
)
from torch_geometric.nn import radius_graph  # noqa: E402

print("repo_root =", REPO_ROOT)
print("capsule   =", CAPSULE_CODE)


repo_root = /Users/rahul/Desktop/hp-proteins-ml
capsule   = /Users/rahul/Desktop/hp-proteins-ml/capsule-3259363/code


In [3]:
# Paths and runtime config
EXPERIMENTAL_CSV = REPO_ROOT / "ramanchembl_pipeline" / "experimental_data" / "raman_spectra_db.csv"
METADATA_CSV = REPO_ROOT / "ramanchembl_pipeline" / "experimental_data" / "metadata_db.csv"
METADATA_URL = "https://raw.githubusercontent.com/mteranm/ramanbiolib/main/ramanbiolib/db/metadata_db.csv"

# Depolar path must match depolar_spectra_pipeline.ipynb
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "spectra_queue" / "prodq-depolar-a100x8-20260219-044935"

# Candidate checkpoints (selected by compatibility, not just existence).
WEIGHT_PATHS = {
    "Hi": [
        REPO_ROOT / "artifacts" / "hi" / "prod-hi-a10080x8-clean-20260224-182057" / "latest_Hi.pth",
        REPO_ROOT / "artifacts" / "hi" / "prod-hi-a10080x8-clean-20260224-182057" / "latest_Hi_from_prod-hi-a10080x8-clean-20260224-132537.pth",
        REPO_ROOT / "artifacts" / "hi" / "prod-hi-a10080x8-clean-20260224-182057" / "latest_Hi_state.pth",
        REPO_ROOT / "artifacts" / "hi" / "prod-hi-a10080x8-clean-20260224-182057" / "latest_Hi_state_from_prod-hi-a10080x8-clean-20260224-132537.pth",
    ],
    "Hij": [
        REPO_ROOT / "artifacts" / "hessian" / "latest_Hij.pth",
        REPO_ROOT / "artifacts" / "hessian" / "latest_Hij_epoch1.pth",
        REPO_ROOT / "artifacts" / "hessian" / "latest_Hij_epoch0.pth",
        REPO_ROOT / "artifacts" / "hessian" / "latest_Hij_state.pth",
        REPO_ROOT / "artifacts" / "hij" / "prod-hij-a10080x8-2ep-20260224-232300" / "latest_Hij.pth",
    ],
    "polar": [
        REPO_ROOT / "artifacts" / "polar" / "polar-prod-standard-ckpt-a100x8-20260218-235335" / "latest_polar.pth",
    ],
}

DEDIPOLE_WEIGHTS = REPO_ROOT / "artifacts" / "dedipole" / "prod-dedipole-a10080x8-2ep-20260225-040142" / "latest_dedipole.pth"
DIPOLE_WEIGHTS = REPO_ROOT / "artifacts" / "dipole" / "dipole-prod-standard-ckpt-a100x8-20260219-034602-pt3" / "latest_dipole.pth"

OUT_DIR = REPO_ROOT / "ramanchembl_pipeline" / "artifacts" / "stats"
CACHE_DIR = REPO_ROOT / "ramanchembl_pipeline" / "artifacts" / "stats_cache"
RESOLVER_CACHE_JSON = CACHE_DIR / "component_structure_cache.json"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"  # switch to "cuda" if available and desired

# Initial run controls (set MAX_EVAL_ROWS=None to process all resolvable rows)
MAX_EVAL_ROWS = None
MAX_ATOMS_FOR_INFERENCE = 120

# Spectral simulation settings (aligned with depolar_spectra_pipeline patterns)
SIGMA = 12.0
TEMP = 298.0
INIT_WL = 532.0
FREQ_SCALE_FACTOR = 0.967  # single-pass frequency scaling
FRECHET_STRIDE = 5
X_MIN, X_MAX, N_POINTS = 500.0, 4000.0, 3501

# Name-resolution settings
REQUEST_SLEEP_SECONDS = 0.05
FORCE_REFRESH_UNRESOLVED = False

print("experimental_csv =", EXPERIMENTAL_CSV)
print("depolar_artifact =", ARTIFACT_DIR)
print("weight_paths keys=", sorted(WEIGHT_PATHS))
print("dedipole_weights =", DEDIPOLE_WEIGHTS)
print("dipole_weights   =", DIPOLE_WEIGHTS)
print("out_dir          =", OUT_DIR)

experimental_csv = /Users/rahul/Desktop/hp-proteins-ml/ramanchembl_pipeline/experimental_data/raman_spectra_db.csv
depolar_artifact = /Users/rahul/Desktop/hp-proteins-ml/artifacts/spectra_queue/prodq-depolar-a100x8-20260219-044935
weight_paths keys= ['Hi', 'Hij', 'polar']
dedipole_weights = /Users/rahul/Desktop/hp-proteins-ml/artifacts/dedipole/prod-dedipole-a10080x8-2ep-20260225-040142/latest_dedipole.pth
dipole_weights   = /Users/rahul/Desktop/hp-proteins-ml/artifacts/dipole/dipole-prod-standard-ckpt-a100x8-20260219-034602-pt3/latest_dipole.pth
out_dir          = /Users/rahul/Desktop/hp-proteins-ml/ramanchembl_pipeline/artifacts/stats


In [4]:
# Core helpers (parsing + metrics)
def parse_float_list(value: Any) -> np.ndarray:
    if isinstance(value, list):
        arr = np.asarray(value, dtype=np.float64)
    elif isinstance(value, str):
        arr = np.asarray(ast.literal_eval(value), dtype=np.float64)
    else:
        arr = np.asarray(value, dtype=np.float64)
    return arr


def normalize_signal(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float64)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    y = np.clip(y, 0.0, None)
    m = float(np.max(y)) if y.size else 0.0
    if m <= 0:
        return np.zeros_like(y)
    return y / m


def lines_to_norm_spectrum(freq, activity, x_grid, sigma=12.0, temp=298.0, init_wl=532.0):
    freq = np.asarray(freq, dtype=np.float64)
    activity = np.asarray(activity, dtype=np.float64)

    valid = np.isfinite(freq) & np.isfinite(activity) & (freq > 1e-8)
    freq = freq[valid]
    activity = activity[valid]
    if freq.size == 0:
        return np.zeros_like(x_grid, dtype=np.float64)

    x_t = torch.as_tensor(x_grid, dtype=torch.float64)
    f_t = torch.as_tensor(freq, dtype=torch.float64)
    a_t = torch.as_tensor(activity, dtype=torch.float64)

    broadened = Lorenz_broadening(f_t, a_t, c=x_t, sigma=float(sigma))
    spec = get_raman_intensity(x_t, broadened, temp=float(temp), init_wl=float(init_wl)).detach().cpu().numpy()
    return normalize_signal(spec)


def resample_experimental_to_grid(x_exp, y_exp, x_grid):
    x_exp = np.asarray(x_exp, dtype=np.float64)
    y_exp = np.asarray(y_exp, dtype=np.float64)

    valid = np.isfinite(x_exp) & np.isfinite(y_exp)
    x_exp = x_exp[valid]
    y_exp = y_exp[valid]
    if x_exp.size < 3:
        return np.zeros_like(x_grid, dtype=np.float64)

    order = np.argsort(x_exp)
    x_exp = x_exp[order]
    y_exp = y_exp[order]

    x_u, idx = np.unique(x_exp, return_index=True)
    y_u = y_exp[idx]
    if x_u.size < 3:
        return np.zeros_like(x_grid, dtype=np.float64)

    y_grid = np.interp(x_grid, x_u, y_u, left=0.0, right=0.0)
    return normalize_signal(y_grid)


def discrete_frechet_distance(curve_a: np.ndarray, curve_b: np.ndarray) -> float:
    a = np.asarray(curve_a, dtype=np.float64)
    b = np.asarray(curve_b, dtype=np.float64)
    if a.ndim != 2 or b.ndim != 2 or a.shape[1] != 2 or b.shape[1] != 2:
        raise ValueError("curves must be shaped [N,2] and [M,2]")
    n, m = a.shape[0], b.shape[0]
    if n == 0 or m == 0:
        return float("nan")

    d = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
    ca = np.empty((n, m), dtype=np.float64)

    ca[0, 0] = d[0, 0]
    for i in range(1, n):
        ca[i, 0] = max(ca[i - 1, 0], d[i, 0])
    for j in range(1, m):
        ca[0, j] = max(ca[0, j - 1], d[0, j])
    for i in range(1, n):
        for j in range(1, m):
            ca[i, j] = max(min(ca[i - 1, j], ca[i - 1, j - 1], ca[i, j - 1]), d[i, j])
    return float(ca[n - 1, m - 1])



In [5]:
# Name-resolution + PubChem 3D geometry helpers
PTE_SYMBOLS = [
    "X", "H", "He", "Li", "Be", "B", "C", "N", "O", "F", "Ne", "Na", "Mg", "Al", "Si", "P", "S",
    "Cl", "Ar", "K", "Ca", "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Ga", "Ge", "As",
    "Se", "Br", "Kr", "Rb", "Sr", "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd", "In", "Sn",
    "Sb", "Te", "I", "Xe", "Cs", "Ba", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho",
    "Er", "Tm", "Yb", "Lu", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg", "Tl", "Pb", "Bi", "Po",
    "At", "Rn", "Fr", "Ra", "Ac", "Th", "Pa", "U", "Np", "Pu", "Am", "Cm", "Bk", "Cf", "Es", "Fm", "Md",
    "No", "Lr", "Rf", "Db", "Sg", "Bh", "Hs", "Mt", "Ds", "Rg", "Cn", "Nh", "Fl", "Mc", "Lv", "Ts", "Og",
]
SYMBOL_TO_Z = {s: i for i, s in enumerate(PTE_SYMBOLS) if i > 0}


def make_http_session() -> requests.Session:
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session = requests.Session()
    session.headers.update({"User-Agent": "hp-proteins-ml-calibration/0.1"})
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def normalize_component_name(name: str) -> str:
    s = str(name).strip()
    # Normalize common unicode artifacts from source tables.
    s = s.replace("ﬂ", "fl").replace("α", "alpha").replace("β", "beta")
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s)
    return s


def component_name_candidates(name: str) -> list[str]:
    base = normalize_component_name(name)
    candidates = [base]

    # Common stereochemistry prefixes from experimental labels.
    for prefix in ["d-(+)-", "d-(-)-", "l-(+)-", "l-(-)-", "d-", "l-"]:
        if base.lower().startswith(prefix):
            candidates.append(base[len(prefix):])

    # Fix missing space before "acid" variants: "myristicacid" -> "myristic acid".
    candidates.append(re.sub(r"([A-Za-z])acid\b", r"\1 acid", base))

    # Remove extra spaces around hyphens.
    candidates.append(re.sub(r"\s*-\s*", "-", base))

    # Alternate alpha notation occasionally used by registries.
    candidates.append(base.replace("alpha", "a"))

    # De-duplicate while preserving order.
    out = []
    seen = set()
    for c in candidates:
        c = re.sub(r"\s+", " ", c).strip()
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    return out


def fetch_pubchem_name_record(name: str, session: requests.Session) -> Optional[dict]:
    quoted = requests.utils.quote(name, safe="")
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        f"{quoted}/property/SMILES,ConnectivitySMILES/JSON"
    )
    resp = session.get(url, timeout=30)
    if resp.status_code != 200:
        return None
    try:
        prop = resp.json()["PropertyTable"]["Properties"][0]
        return {
            "cid": int(prop["CID"]),
            "canonical_smiles": prop.get("ConnectivitySMILES") or prop.get("SMILES"),
            "isomeric_smiles": prop.get("SMILES") or prop.get("ConnectivitySMILES"),
        }
    except Exception:
        return None


def fetch_pubchem_sdf_3d(cid: int, session: requests.Session) -> Optional[str]:
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/SDF?record_type=3d"
    # Explicit retries here because PubChem occasionally returns transient 503.
    for attempt in range(1, 6):
        resp = session.get(url, timeout=30)
        if resp.status_code == 200 and "V2000" in resp.text[:400]:
            return resp.text
        if resp.status_code in (404,):
            return None
        time.sleep(min(2.0, 0.25 * attempt))
    return None


def parse_sdf_v2000_geometry(sdf_text: str) -> Optional[tuple[list[list[float]], list[int]]]:
    lines = sdf_text.splitlines()
    if len(lines) < 5:
        return None

    counts = lines[3]
    if len(counts) < 6:
        return None

    try:
        n_atoms = int(counts[0:3])
    except Exception:
        return None

    start = 4
    end = start + n_atoms
    if len(lines) < end:
        return None

    pos: list[list[float]] = []
    z: list[int] = []

    for line in lines[start:end]:
        parts = line.split()
        if len(parts) < 4:
            return None
        try:
            x = float(parts[0])
            y = float(parts[1])
            zc = float(parts[2])
        except Exception:
            return None

        symbol = parts[3]
        znum = SYMBOL_TO_Z.get(symbol)
        if znum is None:
            return None

        pos.append([x, y, zc])
        z.append(int(znum))

    return pos, z


def resolve_component_to_structure(component: str, session: requests.Session) -> dict:
    for cand in component_name_candidates(component):
        rec = fetch_pubchem_name_record(cand, session)
        if not rec:
            continue

        sdf = fetch_pubchem_sdf_3d(rec["cid"], session)
        if not sdf:
            continue

        geom = parse_sdf_v2000_geometry(sdf)
        if not geom:
            continue

        pos, z = geom
        return {
            "status": "resolved",
            "source": "pubchem",
            "query_name": cand,
            "cid": rec["cid"],
            "canonical_smiles": rec.get("canonical_smiles"),
            "isomeric_smiles": rec.get("isomeric_smiles"),
            "n_atoms": len(z),
            "pos": pos,
            "z": z,
        }

    return {
        "status": "unresolved",
        "source": "pubchem",
        "query_name": None,
        "cid": None,
        "canonical_smiles": None,
        "isomeric_smiles": None,
        "n_atoms": None,
        "pos": None,
        "z": None,
    }


In [6]:
# Load experimental data and metadata; then resolve component -> structure with local cache.
if not METADATA_CSV.exists():
    print("Downloading metadata_db.csv from RamanBioLib...")
    md = requests.get(METADATA_URL, timeout=60)
    md.raise_for_status()
    METADATA_CSV.write_bytes(md.content)

exp_df = pd.read_csv(EXPERIMENTAL_CSV)
meta_df = pd.read_csv(METADATA_CSV)

exp_df["wavenumbers_arr"] = exp_df["wavenumbers"].apply(parse_float_list)
exp_df["intensity_arr"] = exp_df["intensity"].apply(parse_float_list)

components = sorted(exp_df["component"].astype(str).unique())

if RESOLVER_CACHE_JSON.exists():
    resolver_cache = json.loads(RESOLVER_CACHE_JSON.read_text())
else:
    resolver_cache = {}

session = make_http_session()
resolved_now = 0
checked = 0

for comp in components:
    checked += 1
    cached = resolver_cache.get(comp)
    if cached is not None:
        if FORCE_REFRESH_UNRESOLVED and cached.get("status") == "unresolved":
            pass
        else:
            continue

    resolver_cache[comp] = resolve_component_to_structure(comp, session)
    resolved_now += int(resolver_cache[comp].get("status") == "resolved")
    if checked % 20 == 0:
        print(f"checked {checked}/{len(components)} components")
    time.sleep(REQUEST_SLEEP_SECONDS)

RESOLVER_CACHE_JSON.write_text(json.dumps(resolver_cache, indent=2))

resolution_rows = []
for comp in components:
    r = resolver_cache.get(comp, {})
    resolution_rows.append(
        {
            "component": comp,
            "status": r.get("status"),
            "source": r.get("source"),
            "query_name": r.get("query_name"),
            "cid": r.get("cid"),
            "canonical_smiles": r.get("canonical_smiles"),
            "n_atoms": r.get("n_atoms"),
        }
    )

resolution_df = pd.DataFrame(resolution_rows)
resolution_csv = OUT_DIR / "component_resolution.csv"
resolution_df.to_csv(resolution_csv, index=False)

print("components_total    =", len(components))
print("resolved_components =", int((resolution_df["status"] == "resolved").sum()))
print("unresolved_components =", int((resolution_df["status"] != "resolved").sum()))
print("newly_resolved_this_run =", resolved_now)
print("resolution_csv =", resolution_csv)
resolution_df.head(12)


components_total    = 141
resolved_components = 76
unresolved_components = 65
newly_resolved_this_run = 0
resolution_csv = /Users/rahul/Desktop/hp-proteins-ml/ramanchembl_pipeline/artifacts/stats/component_resolution.csv


,component,status,source,query_name,cid,canonical_smiles,n_atoms
0,12-methyltetradecanoic acid,resolved,pubchem,12-methyltetradecanoic acid,21672.0,None,47.0
1,13-methylmyristicacid,resolved,pubchem,13-methylmyristic acid,151014.0,None,47.0
2,14-methylhexadecanoic acid,resolved,pubchem,14-methylhexadecanoic acid,22207.0,None,53.0
3,14-methylpentadecanoic acid,resolved,pubchem,14-methylpentadecanoic acid,36247.0,None,50.0
4,15-methylpalmiticacid,resolved,pubchem,15-methylpalmitic acid,164860.0,None,53.0
5,2-deoxy-d-ribose,resolved,pubchem,2-deoxy-d-ribose,5460005.0,None,19.0
6,a-dna,unresolved,pubchem,NaN,NaN,None,NaN
7,acetoacetate,resolved,pubchem,acetoacetate,6971017.0,None,12.0
8,acetyl coenzyme a,unresolved,pubchem,NaN,NaN,None,NaN
9,adenine,resolved,pubchem,adenine,190.0,None,15.0


In [7]:
# Load depolar/Hi/Hij models for actual Raman inference.
def existing_ckpts(candidates):
    return [c for c in candidates if c.exists()]


def _build_args_from_config(cfg: dict, device: str, task_override: str | None = None):
    import argparse

    task_name = task_override or cfg.get("task", "depolar")
    return argparse.Namespace(
        task=task_name,
        num_features=cfg.get("num_features", 160),
        num_block=cfg.get("num_block", 4),
        num_radial=cfg.get("num_radial", 32),
        attention_head=cfg.get("attention_head", 8),
        rc=cfg.get("rc", 5.0),
        dropout=cfg.get("dropout", 0.1),
        pre_layernorm=cfg.get("pre_layernorm", True),
        pre_layernorm_eps=cfg.get("pre_layernorm_eps", 1e-5),
        elora_path=cfg.get("elora_path", "vendored"),
        device=device,
        use_adalora=cfg.get("use_adalora", True),
        adalora_r=cfg.get("adalora_r", 256),
        adalora_alpha=cfg.get("adalora_alpha", 512),
        adalora_dropout=cfg.get("adalora_dropout", 0.1),
        adalora_tinit=cfg.get("adalora_tinit", 10),
        adalora_tfinal=cfg.get("adalora_tfinal", 20),
        adalora_total_step=cfg.get("adalora_total_step", 1000),
        adalora_target_r=cfg.get("adalora_target_r", 128),
        adalora_rslora=cfg.get("adalora_rslora", True),
        adalora_targets=cfg.get("adalora_targets", None),
        adalora_scalar_heads=cfg.get("adalora_scalar_heads", True),
        adalora_attention=cfg.get("adalora_attention", True),
        adalora_all_linears=cfg.get("adalora_all_linears", True),
        adapter_unfreeze_initial=cfg.get("adapter_unfreeze_initial", True),
        adapter_unfreeze_prefixes=cfg.get("adapter_unfreeze_prefixes", None),
        adapter_freeze_base=cfg.get("adapter_freeze_base", True),
    )


def _extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ("model", "state_dict", "module"):
            if key in obj and isinstance(obj[key], dict):
                obj = obj[key]
                break
    if not isinstance(obj, dict):
        raise TypeError(f"checkpoint payload is not dict-like: {type(obj)}")
    if any(k.startswith("module.") for k in obj.keys()):
        obj = {k.replace("module.", "", 1): v for k, v in obj.items()}
    return obj


def _build_and_try_load(task: str, ckpt_path: Path, base_cfg: dict, device: str):
    args = _build_args_from_config(base_cfg, device=device, task_override=task)
    model = build_model(args)
    loaded = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = _extract_state_dict(loaded)
    missing, unexpected = model.load_state_dict(state, strict=False)
    model.eval()
    return model, missing, unexpected


def load_model_from_candidates(task: str, candidates: list[Path], base_cfg: dict, device: str):
    tried = []
    for ckpt in existing_ckpts(candidates):
        try:
            model, missing, unexpected = _build_and_try_load(task, ckpt, base_cfg, device)
            print(f"[{task}] selected {ckpt}")
            print(f"[{task}] missing={len(missing)} unexpected={len(unexpected)}")
            if missing:
                print(f"[{task}] first missing: {missing[:5]}")
            if unexpected:
                print(f"[{task}] first unexpected: {unexpected[:5]}")
            return model, ckpt
        except Exception as e:
            tried.append((ckpt, str(e).split("\n", 1)[0]))
            continue
    detail = "\n".join([f" - {p}: {msg}" for p, msg in tried])
    raise RuntimeError(f"No compatible checkpoint for task={task}. Tried:\n{detail}")


def load_depolar_with_cfg(artifact_dir: Path, device: str) -> tuple[torch.nn.Module, dict]:
    config_path = artifact_dir / "config.json"
    weights_path = artifact_dir / "latest_depolar.pth"
    if not config_path.exists():
        raise FileNotFoundError(f"missing config: {config_path}")
    if not weights_path.exists():
        raise FileNotFoundError(f"missing weights: {weights_path}")

    cfg = json.loads(config_path.read_text())
    args = _build_args_from_config(cfg, device=device, task_override="depolar")
    model = build_model(args)

    loaded = torch.load(weights_path, map_location=device, weights_only=False)
    state = _extract_state_dict(loaded)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[depolar] loaded {weights_path}")
    print(f"[depolar] missing={len(missing)} unexpected={len(unexpected)}")
    if missing:
        print(f"[depolar] first missing: {missing[:5]}")
    if unexpected:
        print(f"[depolar] first unexpected: {unexpected[:5]}")

    model.eval()
    return model, cfg


device_t = torch.device(DEVICE)
depolar_model, depolar_cfg = load_depolar_with_cfg(ARTIFACT_DIR, DEVICE)
hi_model, hi_ckpt = load_model_from_candidates("Hi", WEIGHT_PATHS["Hi"], depolar_cfg, DEVICE)
hij_model, hij_ckpt = load_model_from_candidates("Hij", WEIGHT_PATHS["Hij"], depolar_cfg, DEVICE)

# Loaded for completeness/path validation (not used in Raman metric path here).
dedipole_model, dedipole_ckpt = load_model_from_candidates("dedipole", [DEDIPOLE_WEIGHTS], depolar_cfg, DEVICE)
dipole_model, dipole_ckpt = load_model_from_candidates("dipole", [DIPOLE_WEIGHTS], depolar_cfg, DEVICE)

print("models loaded on", device_t)
print("selected_hi_ckpt =", hi_ckpt)
print("selected_hij_ckpt=", hij_ckpt)



/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(


[depolar] loaded /Users/rahul/Desktop/hp-proteins-ml/artifacts/spectra_queue/prodq-depolar-a100x8-20260219-044935/latest_depolar.pth
[depolar] missing=0 unexpected=0


/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(


[Hi] selected /Users/rahul/Desktop/hp-proteins-ml/artifacts/hi/prod-hi-a10080x8-clean-20260224-182057/latest_Hi.pth
[Hi] missing=0 unexpected=0


/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(
/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(
/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(
/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instea

[Hij] selected /Users/rahul/Desktop/hp-proteins-ml/artifacts/hij/prod-hij-a10080x8-2ep-20260224-232300/latest_Hij.pth
[Hij] missing=0 unexpected=0


/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(


[dedipole] selected /Users/rahul/Desktop/hp-proteins-ml/artifacts/dedipole/prod-dedipole-a10080x8-2ep-20260225-040142/latest_dedipole.pth
[dedipole] missing=0 unexpected=0


/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(


[dipole] selected /Users/rahul/Desktop/hp-proteins-ml/artifacts/dipole/dipole-prod-standard-ckpt-a100x8-20260219-034602-pt3/latest_dipole.pth
[dipole] missing=0 unexpected=0
models loaded on cpu
selected_hi_ckpt = /Users/rahul/Desktop/hp-proteins-ml/artifacts/hi/prod-hi-a10080x8-clean-20260224-182057/latest_Hi.pth
selected_hij_ckpt= /Users/rahul/Desktop/hp-proteins-ml/artifacts/hij/prod-hij-a10080x8-2ep-20260224-232300/latest_Hij.pth


In [8]:
import sys
import importlib

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import ramanchembl_pipeline.stats_notebook_lib as stats_notebook_lib

importlib.reload(stats_notebook_lib)
run_raman_stats_analysis = stats_notebook_lib.run_raman_stats_analysis


X_GRID = np.linspace(float(X_MIN), float(X_MAX), int(N_POINTS), dtype=np.float64)


def predict_raman_from_geometry(pos, z, x_grid):
    pos_t = torch.tensor(pos, dtype=torch.float32, device=device_t, requires_grad=True)
    z_t = torch.tensor(z, dtype=torch.long, device=device_t)

    edge_index = radius_graph(x=pos_t, r=5.0)

    with torch.enable_grad():
        hi = hi_model(pos=pos_t, z=z_t)
        hij = hij_model(pos=pos_t, z=z_t, edge_index=edge_index)
        dp = depolar_model(z=z_t, pos=pos_t)

        freq, modes = hessfreq(
            Hi=hi,
            Hij=hij,
            edge_index=edge_index,
            masses=atom_masses[z_t],
            normal=False,
            linear=False,
            scale=1.0,
        )
        raman_act = get_raman_act(chain_rule_raman(dp=dp, modes=modes))

    freq = torch.nan_to_num(freq, nan=0.0, posinf=0.0, neginf=0.0)
    raman_act = torch.nan_to_num(raman_act, nan=0.0, posinf=0.0, neginf=0.0)

    freq_np = freq.detach().cpu().numpy().astype(np.float64)
    act_np = raman_act.detach().cpu().numpy().astype(np.float64)

    valid = np.isfinite(freq_np) & np.isfinite(act_np) & (freq_np > 1e-8)
    freq_np = freq_np[valid] * float(FREQ_SCALE_FACTOR)
    act_np = act_np[valid]

    y_pred = lines_to_norm_spectrum(
        freq_np,
        act_np,
        x_grid,
        sigma=float(SIGMA),
        temp=float(TEMP),
        init_wl=float(INIT_WL),
    )
    return y_pred, freq_np, act_np


# Alignment Correction Study

Freeze the base Raman predictor, cache paired `predicted -> target` spectra and extracted peaks for the experimental and RamanChemBL/DFT subsets, then compare transport warping, deterministic peak calibration, and direct peak-level `joint_l2` correction conditioned on molecule identity strings. The notebook runs locally first and prints a Modal handoff message only if the projected training time exceeds the local budget.


In [ ]:
from IPython.display import Markdown, display
import os
import importlib

import ramanchembl_pipeline.alignment_notebook_lib as alignment_notebook_lib

importlib.reload(alignment_notebook_lib)

USE_MODAL_VOLUME = bool(int(os.environ.get("ALIGNMENT_USE_MODAL_VOLUME", "0")))
MODAL_VOLUME_ROOT = Path(os.environ.get("ALIGNMENT_MODAL_VOLUME_ROOT", "/mnt/raman-alignment"))
ALIGNMENT_ROOT = (MODAL_VOLUME_ROOT if USE_MODAL_VOLUME else (REPO_ROOT / "ramanchembl_pipeline" / "artifacts" / "alignment"))
ALIGNMENT_CACHE_DIR = ALIGNMENT_ROOT / "cache"

ALIGNMENT_EXPERIMENTAL_MAX_ROWS = int(os.environ.get("ALIGNMENT_EXPERIMENTAL_MAX_ROWS", "202"))
ALIGNMENT_DFT_MAX_CASES         = int(os.environ.get("ALIGNMENT_DFT_MAX_CASES", "1000"))
ALIGNMENT_REFRESH_DATASETS      = bool(int(os.environ.get("ALIGNMENT_REFRESH_DATASETS", "0")))
ALIGNMENT_DEVICE                = os.environ.get("ALIGNMENT_DEVICE", DEVICE)
ALIGNMENT_TIME_BUDGET_MINUTES   = float(os.environ.get("ALIGNMENT_TIME_BUDGET_MINUTES", "20"))

ALIGNMENT_ROOT.mkdir(parents=True, exist_ok=True)
ALIGNMENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# AlignmentTrainConfig v2 — all fields env-var overrideable
ALIGNMENT_TRAIN_CONFIG = alignment_notebook_lib.AlignmentTrainConfig(
    batch_size          = int(os.environ.get("ALIGNMENT_BATCH_SIZE",       "32")),
    max_epochs          = int(os.environ.get("ALIGNMENT_MAX_EPOCHS",       "120")),
    patience            = int(os.environ.get("ALIGNMENT_PATIENCE",         "20")),
    lr                  = float(os.environ.get("ALIGNMENT_LR",             "3e-4")),
    weight_decay        = float(os.environ.get("ALIGNMENT_WEIGHT_DECAY",   "1e-3")),
    latent_dim          = int(os.environ.get("ALIGNMENT_LATENT_DIM",       "128")),
    mol_latent_dim      = int(os.environ.get("ALIGNMENT_MOL_LATENT_DIM",   "64")),
    transformer_heads   = int(os.environ.get("ALIGNMENT_TRANSFORMER_HEADS","8")),
    transformer_layers  = int(os.environ.get("ALIGNMENT_TRANSFORMER_LAYERS","4")),
    string_feature_dim  = int(os.environ.get("ALIGNMENT_STRING_FEATURE_DIM","128")),
    max_freq_delta      = float(os.environ.get("ALIGNMENT_MAX_FREQ_DELTA", "150.0")),
    coverage_loss_weight= float(os.environ.get("ALIGNMENT_COV_LOSS_WEIGHT","2.0")),
    coverage_target_cm  = float(os.environ.get("ALIGNMENT_COV_TARGET_CM", "10.0")),
    confidence_loss_weight = float(os.environ.get("ALIGNMENT_CONF_LOSS_WEIGHT", "1.0")),
    confidence_threshold   = float(os.environ.get("ALIGNMENT_CONF_THRESHOLD", "0.5")),
    repulsion_loss_weight  = float(os.environ.get("ALIGNMENT_REPULSION_WEIGHT", "0.5")),
    repulsion_radius_cm    = float(os.environ.get("ALIGNMENT_REPULSION_RADIUS", "5.0")),
)

print("alignment_root       =", ALIGNMENT_ROOT)
print("alignment_cache_dir  =", ALIGNMENT_CACHE_DIR)
print("alignment_device     =", ALIGNMENT_DEVICE)
print("dft_max_cases        =", ALIGNMENT_DFT_MAX_CASES)
print("refresh_datasets     =", ALIGNMENT_REFRESH_DATASETS)
print(ALIGNMENT_TRAIN_CONFIG)

In [10]:
experimental_alignment_dataset = alignment_notebook_lib.build_experimental_alignment_dataset(
    exp_df=exp_df,
    resolver_cache=resolver_cache,
    predict_fn=predict_raman_from_geometry,
    x_grid=X_GRID,
    cache_dir=ALIGNMENT_CACHE_DIR,
    max_rows=ALIGNMENT_EXPERIMENTAL_MAX_ROWS,
    max_atoms=MAX_ATOMS_FOR_INFERENCE,
    refresh=ALIGNMENT_REFRESH_DATASETS,
)

dft_alignment_dataset = alignment_notebook_lib.build_dft_mode_alignment_dataset(
    db_path=REPO_ROOT / "ramanchembl_pipeline" / "dataset" / "molecule.db",
    predict_fn=predict_raman_from_geometry,
    x_grid=X_GRID,
    lines_to_spectrum_fn=lambda freq, inten, xg: lines_to_norm_spectrum(freq, inten, xg, sigma=float(SIGMA), temp=float(TEMP), init_wl=float(INIT_WL)),
    cache_dir=ALIGNMENT_CACHE_DIR,
    max_cases=ALIGNMENT_DFT_MAX_CASES,
    sample_seed=ALIGNMENT_TRAIN_CONFIG.seed,
    pred_freq_scale_factor=1.0,  # predict_raman_from_geometry already applies FREQ_SCALE_FACTOR
    refresh=ALIGNMENT_REFRESH_DATASETS,
)

print("experimental_cases =", len(experimental_alignment_dataset))
print("dft_cases          =", len(dft_alignment_dataset))
display(experimental_alignment_dataset.metadata.head())
display(dft_alignment_dataset.metadata.head())

experimental_cases = 89
dft_cases          = 1000


,component,cid,n_atoms
0,12-methyltetradecanoic acid,21672,47
1,13-methylmyristicacid,151014,47
2,14-methylhexadecanoic acid,22207,53
3,14-methylpentadecanoic acid,36247,50
4,15-methylpalmiticacid,164860,53


,molecule_id,smiles
0,158,CCCCCCCCCCCCCCOc1ccccc1C(=O)OC
1,447,CCO/C(=C/1\C(=NC(=C([C@H]1c1cccc(c1)n1ccnc1)C(...
2,453,NCCCC[C@@H](C(=O)N)NC(=O)[C@@H](NC(=O)[C@H](Cc...
3,514,Clc1ccc(cc1)n1nc(c2c1c1ccccc1oc2=O)C
4,690,COCCN([C]1N=C(C)[N]c2c1nnn2c1ccc(cc1Br)C(C)C)CCOC


In [11]:
dft_minutes = alignment_notebook_lib._runtime_estimate_minutes(dft_alignment_dataset, ALIGNMENT_TRAIN_CONFIG, ALIGNMENT_DEVICE)
print(f"projected dft train time: {dft_minutes:.1f} min")
if dft_minutes > ALIGNMENT_TIME_BUDGET_MINUTES:
    display(Markdown(alignment_notebook_lib.modal_notebook_guidance()))

projected dft train time: 60.0 min


Projected runtime high. Use `ALIGNMENT_USE_MODAL_VOLUME=1` at /mnt/raman

In [12]:
alignment_results = alignment_notebook_lib.run_alignment_study(
    experimental_dataset=experimental_alignment_dataset,
    dft_dataset=dft_alignment_dataset,
    out_dir=ALIGNMENT_ROOT,
    device=ALIGNMENT_DEVICE,
    train_config=ALIGNMENT_TRAIN_CONFIG,
)

display(Markdown(f"**Checkpoint:** `{alignment_results['checkpoint']}`"))
display(Markdown(f"**Summary CSV:** `{alignment_results['domains']['dft']['summary_csv']}`"))
display(Markdown(alignment_results["domains"]["dft"]["report_markdown"]))

dft_summary = pd.read_csv(alignment_results["domains"]["dft"]["summary_csv"])

DFT_TEST_METRICS = [
    "model", "split", "n_cases",
    "f1@5", "f1@10", "f1@20",
    "coverage@5", "coverage@10",
    "cwmae@5", "cwmae@10",
    "point_rmse", "intensity_mae",
]

display(Markdown("## DFT Alignment Results"))
display(dft_summary[[c for c in DFT_TEST_METRICS if c in dft_summary.columns]])

[21:47:26] Explicit valence for atom # 17 C, 5, is greater than permitted
[21:47:26] Explicit valence for atom # 0 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 11 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 3 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 18 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 13 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 13 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 21 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 5 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 12 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 3 C, 5, is greater than permitted
[21:47:26] Explicit valence for atom # 3 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 2 N, 4, is greater than permitted
[21:47:26] Explicit valence for atom # 3 N, 

Training on cpu | train=700 val=150 test=150


/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Epoch    0 | train=51.3230 val=45.7734 patience=0
Epoch   20 | train=44.2772 val=39.3193 patience=8
Early stopping at epoch 32
Checkpoint saved → /Users/rahul/Desktop/hp-proteins-ml/ramanchembl_pipeline/artifacts/alignment/alignment_model.pth


**Checkpoint:** `/Users/rahul/Desktop/hp-proteins-ml/ramanchembl_pipeline/artifacts/alignment/alignment_model.pth`

**Summary CSV:** `/Users/rahul/Desktop/hp-proteins-ml/ramanchembl_pipeline/artifacts/alignment/dft_alignment_summary.csv`

### DFT Alignment Results (test set)
- F1@5: 0.102  F1@10: 0.170  F1@20: 0.254
- CWMAE@10: 6.22 cm⁻¹  (honest coverage-penalised metric)
- Coverage@10: 0.610  Coverage@5: 0.415
- Point RMSE (matched@10): 5.19 cm⁻¹

## DFT Alignment Results

,model,split,n_cases,f1@5,f1@10,f1@20,coverage@5,coverage@10,cwmae@5,cwmae@10,point_rmse,intensity_mae
0,point_transformer,train,700,0.104305,0.174760,0.258372,0.416019,0.616709,3.829847,6.192943,5.197170,26.161034
1,point_transformer,val,150,0.104990,0.176450,0.257322,0.413767,0.615007,3.837583,6.204441,5.211930,28.701479
2,point_transformer,test,150,0.101897,0.169681,0.253691,0.415224,0.610265,3.834090,6.217795,5.193864,23.812472


In [13]:
case_csv = alignment_results["domains"]["dft"].get("case_csv")
if case_csv and Path(case_csv).exists():
    display(Markdown("## Per-molecule cases (head)"))
    display(pd.read_csv(case_csv).head(10))

## Per-molecule cases (head)

,case_index,model,coverage@5,coverage@10,coverage@15,coverage@20,cwmae@5,cwmae@10,cwmae@15,cwmae@20,f1@5,f1@10,f1@15,f1@20,point_rmse,intensity_mae
0,0,point_transformer,0.344633,0.576271,0.621469,0.649718,4.005602,6.729219,8.746061,10.546701,0.096045,0.141243,0.180791,0.197740,4.563347,2.150510
1,1,point_transformer,0.509434,0.672956,0.805031,0.811321,3.451930,5.453752,6.672287,7.636352,0.127796,0.198083,0.242812,0.274760,4.213233,7.609954
2,2,point_transformer,0.546512,0.740310,0.806202,0.829457,3.233925,4.865176,6.010835,6.922908,0.127237,0.174950,0.214712,0.230616,4.408784,7.436342
3,3,point_transformer,0.387097,0.569892,0.720430,0.795699,3.888833,6.481138,8.143120,9.345306,0.010929,0.065574,0.109290,0.153005,6.551075,5.714930
4,4,point_transformer,0.481481,0.679012,0.765432,0.790123,3.619262,5.578875,6.906655,7.992415,0.092593,0.135802,0.197531,0.228395,4.797845,12.450265
5,5,point_transformer,0.538095,0.719048,0.785714,0.819048,3.397091,5.152108,6.369717,7.326023,0.097324,0.145985,0.194647,0.209246,5.017457,2.544054
6,6,point_transformer,0.505952,0.738095,0.803571,0.839286,3.560084,5.491055,6.611888,7.523207,0.132931,0.199396,0.253776,0.296073,5.218198,3.846577
7,7,point_transformer,0.271605,0.407407,0.506173,0.654321,4.401798,7.730245,10.469179,12.553052,0.049383,0.074074,0.098765,0.148148,5.299962,3.640379
8,8,point_transformer,0.555556,0.736842,0.795322,0.818713,3.459405,5.113142,6.270913,7.254475,0.224242,0.315152,0.345455,0.393939,4.659960,63.583920
9,9,point_transformer,0.608059,0.732601,0.754579,0.765568,3.122187,4.741551,6.011160,7.195085,0.101124,0.172285,0.194757,0.213483,5.643051,1.740951
